In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from torch_geometric.data import HeteroData
import torch

c:\Users\sth3ayush\Desktop\Hackathon\IIMS x Perceptron 2026\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Normalize Drug STITCH

### Handle Imports and Paths

In [2]:
# Project Path
PROJECT_ROOT = Path("..")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
GRAPH_DIR = PROJECT_ROOT / "graph"

PROCESSED_DIR.mkdir(exist_ok=True)
GRAPH_DIR.mkdir(exist_ok=True)

### Reading the datasets

In [3]:
drug_combo = pd.read_csv(RAW_DIR / "drug-combo.csv")
drug_gene = pd.read_csv(RAW_DIR / "drug-gene.csv")
drug_mono = pd.read_csv(RAW_DIR / "drug-mono.csv")
gene_gene = pd.read_csv(RAW_DIR / "gene-gene.csv")
effect_categories = pd.read_csv(RAW_DIR / "effectcategories.csv")

In [ ]:
print("Drug Combo:", drug_combo.shape)
print("Drug Gene:", drug_gene.shape)
print("Drug Mono:", drug_mono.shape)
print("Gene-Gene:", gene_gene.shape)
print("Effect Categories:", effect_categories.shape)

Drug Combo: (4649441, 4)
Drug Gene: (18690, 2)
Drug Mono: (174977, 3)
Gene-Gene: (715612, 2)
Effect Categories: (561, 3)


### Normalizer Function

In [ ]:
def normalize_cid(cid: str, collapse_stereoisomers: bool = True) -> str:
    """
    STITCH convention:
        CID0######## -> "flat" / stereo-collapsed compound
        CID1######## -> a specific stereoisomer of that flat compound
    """
    cid = str(cid).strip()

    if cid.startswith("CID"):
        digits = cid[3:]
        if collapse_stereoisomers and digits.startswith("1") and len(digits) > 1:
            digits = "0" + digits[1:]
        return "CID" + str(int(digits))

    return cid

#### drug-combo normalization

In [ ]:
drug_combo["STITCH 1"] = drug_combo["STITCH 1"].apply(normalize_cid)
drug_combo["STITCH 2"] = drug_combo["STITCH 2"].apply(normalize_cid)

#### drug-gene normalization

In [ ]:
drug_gene["STITCH"] = drug_gene["STITCH"].apply(normalize_cid)

#### drug-mono normalization

In [ ]:
drug_mono["STITCH"] = drug_mono["STITCH"].apply(normalize_cid)

### Save Normalized datasets

In [ ]:
drug_combo.to_csv(
    PROCESSED_DIR / "drug-combo-normalized.csv",
    index=False
)

drug_gene.to_csv(
    PROCESSED_DIR / "drug-gene-normalized.csv",
    index=False
)

drug_mono.to_csv(
    PROCESSED_DIR / "drug-mono-normalized.csv",
    index=False
)

In [ ]:
# Test
drug_combo.head()

,STITCH 1,STITCH 2,Polypharmacy Side Effect,Side Effect Name
0,CID2173,CID3345,C0151714,hypermagnesemia
1,CID2173,CID3345,C0035344,retinopathy of prematurity
2,CID2173,CID3345,C0004144,atelectasis
3,CID2173,CID3345,C0002063,alkalosis
4,CID2173,CID3345,C0004604,Back Ache


## Creating Nodes

In [4]:
NODE_DIR = PROCESSED_DIR / "nodes"

NODE_DIR.mkdir(exist_ok=True)

In [5]:
drug_combo = pd.read_csv(PROCESSED_DIR / "drug-combo-normalized.csv")
drug_gene = pd.read_csv(PROCESSED_DIR / "drug-gene-normalized.csv")
drug_mono = pd.read_csv(PROCESSED_DIR / "drug-mono-normalized.csv")

### Node Table

#### Creating Drug Node

In [6]:
drugs = set()

drugs.update(drug_combo["STITCH 1"])
drugs.update(drug_combo["STITCH 2"])
drugs.update(drug_gene["STITCH"])
drugs.update(drug_mono["STITCH"])

In [7]:
drugs = sorted(drugs)

In [8]:
drug2idx = {
    drug: idx
    for idx, drug in enumerate(drugs)
}

In [9]:
drug_nodes = pd.DataFrame({
    "node_index": range(len(drugs)),
    "drug_id": drugs
})

In [10]:
drug_nodes.to_csv(
    NODE_DIR / "drug_nodes.csv",
    index=False
)

#### Creating Gene Node

In [11]:
genes = set()

genes.update(gene_gene["Gene 1"])
genes.update(gene_gene["Gene 2"])
genes.update(drug_gene["Gene"])

In [12]:
genes = sorted(genes)

In [13]:
gene2idx = {
    gene: idx
    for idx, gene in enumerate(genes)
}

In [14]:
gene_nodes = pd.DataFrame({
    "node_index": range(len(genes)),
    "gene_id": genes
})

In [15]:
gene_nodes.to_csv(
    NODE_DIR / "gene_nodes.csv",
    index=False
)

#### Creating Side Effect Node

In [16]:
side_effects = set()

side_effects.update(drug_mono["Individual Side Effect"])

In [17]:
side_effects = sorted(side_effects)

In [18]:
sideeffect2idx = {
    se: idx
    for idx, se in enumerate(side_effects)
}

In [19]:
sideeffect_nodes = pd.DataFrame({
    "node_index": range(len(side_effects)),
    "side_effect_id": side_effects
})

In [20]:
sideeffect_nodes.to_csv(
    NODE_DIR / "sideeffect_nodes.csv",
    index=False
)

#### Enrich Side Effect Node with Disease Class

In [21]:
se_to_class = dict(zip(
    effect_categories["Side Effect"],
    effect_categories["Disease Class"]
))

sideeffect_nodes["disease_class"] = (
    sideeffect_nodes["side_effect_id"]
    .map(se_to_class)
    .fillna("unknown")
)

sideeffect_nodes.to_csv(
    NODE_DIR / "sideeffect_nodes.csv",
    index=False
)

sideeffect_nodes.head()

,node_index,side_effect_id,disease_class
0,0,C0000727,unknown
1,1,C0000729,unknown
2,2,C0000733,unknown
3,3,C0000734,unknown
4,4,C0000735,unknown


In [22]:
# Test
print(f"Number of drugs: {len(drugs)}")
print(f"Number of genes: {len(genes)}")
print(f"Number of side effects: {len(side_effects)}")

Number of drugs: 70
Number of genes: 19083
Number of side effects: 5004


In [23]:
drug_nodes.head()

,node_index,drug_id
0,0,CID1117
1,1,CID125889
2,2,CID150311
3,3,CID1983
4,4,CID2088


In [24]:
gene_nodes.head()

,node_index,gene_id
0,0,1
1,1,2
2,2,9
3,3,10
4,4,12


In [25]:
sideeffect_nodes.head()

,node_index,side_effect_id,disease_class
0,0,C0000727,unknown
1,1,C0000729,unknown
2,2,C0000733,unknown
3,3,C0000734,unknown
4,4,C0000735,unknown


## Creating Edges

In [26]:
EDGE_DIR = PROCESSED_DIR / "edges"
EDGE_DIR.mkdir(exist_ok=True)

### gene-gene Edge

In [27]:
gene_gene_edges = []

for _, row in gene_gene.iterrows():

    g1 = gene2idx[row["Gene 1"]]
    g2 = gene2idx[row["Gene 2"]]

    # Undirected i.e. storing both directions
    gene_gene_edges.append((g1, g2))
    gene_gene_edges.append((g2, g1))

In [28]:
gene_gene_edges_df = pd.DataFrame(
    gene_gene_edges,
    columns=["source", "target"]
)

In [29]:
gene_gene_edges_df.to_csv(
    EDGE_DIR / "gene_gene_edges.csv",
    index=False
)

In [30]:
gene_gene_edges_df.head()

,source,target
0,14461,17696
1,17696,14461
2,14461,17141
3,17141,14461
4,14461,5158


### drug-gene Edge

In [31]:
drug_gene_edges = []

for _, row in drug_gene.iterrows():

    drug = drug2idx[row["STITCH"]]
    gene = gene2idx[row["Gene"]]

    drug_gene_edges.append((drug, gene))

In [32]:
drug_gene_edges_df = pd.DataFrame(
    drug_gene_edges,
    columns=["drug", "gene"]
)

In [33]:
drug_gene_edges_df.to_csv(
    EDGE_DIR / "drug_gene_edges.csv",
    index=False
)

In [34]:
drug_gene_edges_df.head()

,drug,gene
0,4,8460
1,4,1558
2,4,11437
3,4,8482
4,4,6343


### drug-sideeffect Edge

In [35]:
drug_sideeffect_edges = []

for _, row in drug_mono.iterrows():

    drug = drug2idx[row["STITCH"]]
    side_effect = sideeffect2idx[row["Individual Side Effect"]]

    drug_sideeffect_edges.append((drug, side_effect))

In [36]:
drug_sideeffect_edges_df = pd.DataFrame(
    drug_sideeffect_edges,
    columns=["drug", "side_effect"]
)

In [37]:
drug_sideeffect_edges_df.to_csv(
    EDGE_DIR / "drug_sideeffect_edges.csv",
    index=False
)

In [38]:
drug_sideeffect_edges_df.head()

,drug,side_effect
0,38,1028
1,38,4052
2,38,995
3,38,1579
4,38,1585


In [39]:
# Test
print(f"Gene-Gene edges: {len(gene_gene_edges_df):,}")
print(f"Drug-Gene edges: {len(drug_gene_edges_df):,}")
print(f"Drug-SideEffect edges: {len(drug_sideeffect_edges_df):,}")

Gene-Gene edges: 1,431,224
Drug-Gene edges: 2,522
Drug-SideEffect edges: 20,345


## Polypharmacy Edge

### Relation vocabulary

In [40]:
se_codes = sorted(
    drug_combo["Polypharmacy Side Effect"].unique()
)

# Side Effect ID -> Relation ID
se2relid = {
    se: idx
    for idx, se in enumerate(se_codes)
}

print(f"Number of relation types: {len(se2relid)}")

Number of relation types: 1301


### Build one Drug-Drug edge list

In [41]:
drug_drug_edges = []
edge_types = []

for _, row in drug_combo.iterrows():

    d1 = drug2idx[row["STITCH 1"]]
    d2 = drug2idx[row["STITCH 2"]]

    rel = se2relid[row["Polypharmacy Side Effect"]]

    # Forward edge
    drug_drug_edges.append((d1, d2))
    edge_types.append(rel)

    # Reverse edge (DDIs are symmetric)
    drug_drug_edges.append((d2, d1))
    edge_types.append(rel)

### TO tensors

In [42]:
drug_drug_edge_index = (
    torch.tensor(drug_drug_edges, dtype=torch.long)
    .t()
    .contiguous()
)

drug_drug_edge_type = torch.tensor(
    edge_types,
    dtype=torch.long
)

### Check dimensions and Inspect few edges

In [43]:
print("Edge Index Shape :", drug_drug_edge_index.shape)
print("Edge Type Shape  :", drug_drug_edge_type.shape)

Edge Index Shape : torch.Size([2, 801630])
Edge Type Shape  : torch.Size([801630])


In [44]:
for i in range(5):
    src = drug_drug_edge_index[0, i].item()
    dst = drug_drug_edge_index[1, i].item()
    rel = drug_drug_edge_type[i].item()

    print(
        f"Drug {src} -> Drug {dst}, Relation ID = {rel}"
    )

Drug 56 -> Drug 62, Relation ID = 257
Drug 62 -> Drug 56, Relation ID = 257
Drug 56 -> Drug 62, Relation ID = 1010
Drug 62 -> Drug 56, Relation ID = 1010
Drug 56 -> Drug 62, Relation ID = 807


### Save for reproducibility

In [45]:
edge_df = pd.DataFrame({
    "source": drug_drug_edge_index[0].numpy(),
    "target": drug_drug_edge_index[1].numpy(),
    "relation_id": drug_drug_edge_type.numpy()
})

edge_df.to_csv(
    EDGE_DIR / "drug_drug_polypharmacy_edges.csv",
    index=False
)

In [46]:
relation_df = pd.DataFrame({
    "relation_id": list(se2relid.values()),
    "side_effect": list(se2relid.keys())
})

relation_df.to_csv(
    PROCESSED_DIR / "polypharmacy_relation_lookup.csv",
    index=False
)

### Testing

In [47]:
# Test
print(f"Number of Drug-Drug edges: {drug_drug_edge_index.shape[1]:,}")
print(f"Number of Relation Types : {len(se2relid):,}")

print("\nFirst 5 relations:")
relation_df.head()

Number of Drug-Drug edges: 801,630
Number of Relation Types : 1,301

First 5 relations:


,relation_id,side_effect
0,0,C0000731
1,1,C0000737
2,2,C0000768
3,3,C0000786
4,4,C0000814


In [48]:
print(drug_drug_edge_index.shape)
print(drug_drug_edge_type.shape)

torch.Size([2, 801630])
torch.Size([801630])


In [49]:
print(drug_drug_edge_index[:, :5])
print(drug_drug_edge_type[:5])

tensor([[56, 62, 56, 62, 56],
        [62, 56, 62, 56, 62]])
tensor([ 257,  257, 1010, 1010,  807])


## Create Heterogeneous Graph

### HeteroData Graph and Register node types

In [50]:
data = HeteroData()

In [51]:
data["drug"].num_nodes = len(drugs)
data["gene"].num_nodes = len(genes)
data["side_effect"].num_nodes = len(side_effects)

#### Attach Disease Class as a side_effect node feature

In [52]:
disease_classes = sorted(sideeffect_nodes["disease_class"].unique())
diseaseclass2idx = {c: i for i, c in enumerate(disease_classes)}

data["side_effect"].disease_class = torch.tensor(
    sideeffect_nodes["disease_class"].map(diseaseclass2idx).values,
    dtype=torch.long
)
data["side_effect"].num_disease_classes = len(disease_classes)

In [53]:
print(data)

HeteroData(
  drug={ num_nodes=70 },
  gene={ num_nodes=19083 },
  side_effect={
    num_nodes=5004,
    disease_class=[5004],
    num_disease_classes=1,
  }
)


#### Gene-Gene edges graph

In [54]:
gene_gene_edge_index = torch.tensor(
    gene_gene_edges_df.values.T,
    dtype=torch.long
)

In [55]:
data["gene", "interacts", "gene"].edge_index = gene_gene_edge_index

#### Drug-Gene edges graph

In [56]:
drug_gene_edge_index = torch.tensor(
    drug_gene_edges_df.values.T,
    dtype=torch.long
)

In [57]:
data["drug", "targets", "gene"].edge_index = drug_gene_edge_index

In [58]:
data["gene", "targeted_by", "drug"].edge_index = drug_gene_edge_index.flip(0)

#### Drug-Side Effect edges graph

In [59]:
drug_sideeffect_edge_index = torch.tensor(
    drug_sideeffect_edges_df.values.T,
    dtype=torch.long
)

In [60]:
data["drug", "causes", "side_effect"].edge_index = drug_sideeffect_edge_index

In [61]:
data["side_effect", "caused_by", "drug"].edge_index = (
    drug_sideeffect_edge_index.flip(0)
)

### Polypharmacy edges graph

In [62]:
data["drug", "polypharmacy", "drug"].edge_index = (
    drug_drug_edge_index
)

data["drug", "polypharmacy", "drug"].edge_type = (
    drug_drug_edge_type
)

### Verification

In [63]:
print(data)

HeteroData(
  drug={ num_nodes=70 },
  gene={ num_nodes=19083 },
  side_effect={
    num_nodes=5004,
    disease_class=[5004],
    num_disease_classes=1,
  },
  (gene, interacts, gene)={ edge_index=[2, 1431224] },
  (drug, targets, gene)={ edge_index=[2, 2522] },
  (gene, targeted_by, drug)={ edge_index=[2, 2522] },
  (drug, causes, side_effect)={ edge_index=[2, 20345] },
  (side_effect, caused_by, drug)={ edge_index=[2, 20345] },
  (drug, polypharmacy, drug)={
    edge_index=[2, 801630],
    edge_type=[801630],
  }
)


In [64]:
print(data["drug"])
print(data["gene"])
print(data["side_effect"])

{'num_nodes': 70}
{'num_nodes': 19083}
{'num_nodes': 5004, 'disease_class': tensor([0, 0, 0,  ..., 0, 0, 0]), 'num_disease_classes': 1}


In [65]:
print(data["drug", "polypharmacy", "drug"])

{'edge_index': tensor([[56, 62, 56,  ..., 50, 15, 50],
        [62, 56, 62,  ..., 15, 50, 15]]), 'edge_type': tensor([ 257,  257, 1010,  ...,  362,  432,  432])}


In [66]:
print(data["drug", "targets", "gene"])

{'edge_index': tensor([[    4,     4,     4,  ...,    50,    50,    50],
        [ 8460,  1558, 11437,  ..., 13229,   474,  7796]])}


In [67]:
print(data["gene", "interacts", "gene"])

{'edge_index': tensor([[14461, 17696, 14461,  ...,  3826,  5631,  5636],
        [17696, 14461, 17141,  ...,  3829,  5636,  5631]])}


In [70]:
torch.save(data, GRAPH_DIR / "heterodata.pt")

In [71]:
data = torch.load(
    GRAPH_DIR / "heterodata.pt",
    weights_only=False
)